# Data Cleaning — Air Quality & Low Emission Zone (LEZ) Paris

## Objective
Apply all cleaning decisions made during exploration and save processed datasets to `data/processed/`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import requests

sys.path.insert(0, str(Path().resolve().parent))
from src.data_loader import load_airparif
from src.cleaning import clean_airparif, clean_insee

print("Environment ready ✓")

Environment ready ✓


## 1. Airparif — NO₂ measurements

**Cleaning decisions:**

| Problem | Decision | Rationale |
|---------|----------|-----------|
| 5 stations >20% NaN | Exclude | Cannot contribute reliable trends over the study period |
| Negative values | Replace with NaN | NO₂ < 0 is physically impossible — sensor error |
| Short NaN gaps (≤ 3h) | Linear interpolation | NO₂ varies slowly; short gaps are plausibly sensor glitches |
| Long NaN gaps (> 3h) | Leave as NaN | Interpolating station outages would introduce artificial data |

In [2]:
dfs = {year: load_airparif(year) for year in range(2019, 2024)}
airparif = clean_airparif(dfs)

print(f"Shape: {airparif.shape}")
print(f"NaN rate after cleaning: {airparif.isnull().mean().mean() * 100:.1f}%")

Shape: (43824, 36)
NaN rate after cleaning: 3.8%


In [3]:
# Save to processed
airparif.to_csv("../data/processed/airparif_clean.csv")
print("airparif_clean.csv saved ✓")

airparif_clean.csv saved ✓


**Observations on Airparif cleaning:**
- 5 stations excluded: PA04C, DEF, BASCH, PA01H, SOULT
- 19 negative values (RUR-SO) replaced by NaN
- Remaining NaN interpolated linearly up to 3 consecutive hours
- NaN rate: 4.3% → 3.8% after interpolation (most gaps exceed 3h — station outages)
- AUT shows 8,470 remaining NaN (concentrated in 2023) — retained as it stays below the 20% threshold


## 2. INSEE IRIS — Median income

**Cleaning decisions:**

| Problem | Decision | Rationale |
|---------|----------|-----------|
| Nationwide scope | Filter to depts 75, 92, 93, 94 | Only inner Île-de-France stations are in scope |
| 'ns' / 'nd' codes | Replace with NaN | These are INSEE conventions for suppressed values, not numeric |
| Comma decimals | Convert to dot | Required for float casting |
| 216 NaN after conversion | Keep as NaN | Stations matched to these IRIS will be excluded from income analysis |

In [4]:
insee_raw = pd.read_csv(
    "../data/raw/insee_iris/BASE_TD_FILO_IRIS_2021_DISP.csv",
    sep=";",
    dtype={"IRIS": str}
)
insee = clean_insee(insee_raw)

print(f"Shape: {insee.shape}")
print(f"NaN: {insee['DISP_MED21'].isnull().sum()}")
print(insee.head())

Shape: (2745, 2)
NaN: 216
           IRIS  DISP_MED21
9762  751010101         NaN
9763  751010102         NaN
9764  751010103         NaN
9765  751010104         NaN
9766  751010105         NaN


In [5]:
insee.to_csv("../data/processed/insee_iris_clean.csv", index=False)
print("insee_iris_clean.csv saved ✓")

insee_iris_clean.csv saved ✓


**Observations on INSEE IRIS cleaning:**
- Filtered to 2,745 IRIS covering Paris and inner suburbs (depts 75, 92, 93, 94)
- 216 NaN in DISP_MED21 (170 "ns" — non-significant, 46 "nd" — not available)
- Decimal separator converted from comma to dot
- Note: IRIS codes will need to be matched against an IRIS geometry shapefile for spatial join with Airparif stations (to be handled in analysis)

## 3. Weather data (Open-Meteo)

In [6]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 48.8566,
    "longitude": 2.3522,
    "start_date": "2019-01-01",
    "end_date": "2023-12-31",
    "hourly": "windspeed_10m,precipitation,temperature_2m",
    "timezone": "Europe/Paris"
}

response = requests.get(url, params=params)
meteo_raw = response.json()

meteo = pd.DataFrame(meteo_raw["hourly"])
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.set_index("time")

print(f"Shape: {meteo.shape}")
print(f"NaN: {meteo.isnull().sum().sum()}")

Shape: (43824, 3)
NaN: 0


In [7]:
meteo.to_csv("../data/processed/meteo_clean.csv")
print("meteo_clean.csv saved ✓")

meteo_clean.csv saved ✓


**Observations on Weather data cleaning:**
- 43,824 hourly records, no missing values
- No transformation required — data arrives clean from the API